# Protobuf Basics — 06: Protobuf with Kafka

The most important Protobuf use case in production:
reading and deserializing Protobuf messages from Kafka topics.

Topics: Kafka + Protobuf pattern, Confluent Schema Registry, from_protobuf in streaming,
error handling, deserialize batch Kafka dumps.


In [ ]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/protobuf_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('protobuf-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

## Step 1 — Kafka + Protobuf Architecture



In [ ]:

print("""
Kafka + Protobuf production architecture:

  Producer (gRPC service / Java app)
    Order order = Order.newBuilder()
        .setOrderId(1).setProduct("Laptop").setPrice(999.99)
        .build();
    byte[] bytes = order.toByteArray();
    producer.send(new ProducerRecord("orders", bytes));
  
                    │ binary Protobuf bytes
                    ▼
  Kafka Topic: "orders"
    key:   bytes (optional order_id as bytes)
    value: bytes (serialized Order Protobuf)
  
                    │
                    ▼
  Spark Structured Streaming
    raw = spark.readStream.format("kafka")
               .option("kafka.bootstrap.servers","broker:9092")
               .option("subscribe","orders").load()
    
    orders = raw.select(
        from_protobuf(col("value"), "Order", "/path/orders.desc").alias("o")
    ).select("o.*")
    
    orders.writeStream.format("delta").option("path","/data/orders").start()

  Delta / Iceberg Analytics Zone
    SELECT category, SUM(revenue) FROM orders GROUP BY 1

Confluent Schema Registry variant:
  Producer registers schema → gets schema_id
  Message: [0x00][4-byte schema_id][protobuf bytes]
  Consumer looks up schema by ID → deserializes
""")


## Step 2 — Simulate Kafka Protobuf Messages



In [ ]:

import pathlib, sys, importlib.util, random, struct as st
PROTO_DIR = f"{DATA_DIR}/protos"
pathlib.Path(PROTO_DIR).mkdir(exist_ok=True)

# Use manual encoding from notebook 03
def encode_varint(v):
    r=b""
    while v>0x7F: r+=bytes([(v&0x7F)|0x80]); v>>=7
    return r+bytes([v&0x7F])

def encode_field(fn,wt,val): return encode_varint((fn<<3)|wt)+val
def encode_str(fn,s): e=s.encode(); return encode_field(fn,2,encode_varint(len(e))+e)
def encode_i64(fn,n): return encode_field(fn,0,encode_varint(n))
def encode_dbl(fn,d): return encode_field(fn,1,st.pack('<d',d))

def make_order_proto(order_id,customer_id,product,category,price,quantity,revenue,status):
    m=b""
    m+=encode_i64(1,order_id); m+=encode_i64(2,customer_id)
    m+=encode_str(3,product);  m+=encode_str(4,category)
    m+=encode_dbl(5,price);    m+=encode_i64(6,quantity)
    m+=encode_dbl(7,revenue);  m+=encode_str(8,status)
    return m

def make_kafka_message(schema_id, proto_bytes):
    return bytes([0x00]) + st.pack(">I", schema_id) + proto_bytes

random.seed(42)
CATS=["Electronics","Books","Clothing","Food","Sports"]
messages=[]
for i in range(2000):
    qty=random.randint(1,10); price=round(random.uniform(5,500),2)
    proto=make_order_proto(i+1,random.randint(1,500),f"Product_{i%50}",
                           random.choice(CATS),price,qty,round(price*qty,2),
                           random.choice(["pending","confirmed","shipped"]))
    msg=make_kafka_message(1,proto)
    messages.append((bytearray(msg),))

df_kafka=spark.createDataFrame(messages,["raw_bytes"])
print(f"Simulated {df_kafka.count():,} Kafka messages")
df_kafka.show(3)
print(f"Avg message size: {df_kafka.rdd.map(lambda r: len(r[0])).mean():.0f} bytes")


## Step 3 — Deserialize Kafka Protobuf Messages



In [ ]:

from pyspark.sql.types import *
import struct as st

# UDF to extract schema_id and payload
@F.udf(returnType=LongType())
def get_schema_id(msg):
    if msg and len(msg) >= 5:
        return int(st.unpack(">I", bytes(msg[1:5]))[0])
    return None

@F.udf(returnType=BinaryType())
def get_payload(msg):
    if msg and len(msg) > 5:
        return bytes(msg[5:])
    return None

# Manual decode UDFs (simulating from_protobuf)
@F.udf(returnType=LongType())
def decode_order_id(payload):
    if not payload: return None
    i=0; val=0; shift=0
    b=bytes(payload)
    while i<len(b):
        byte=b[i]; i+=1
        if (byte&0xFC)==0x08:  # field 1, wire type 0 (varint)
            j=i; v=0; s=0
            while j<len(b):
                byt=b[j]; j+=1; v|=(byt&0x7F)<<s; s+=7
                if not (byt&0x80): break
            return v
        break
    return None

df_parsed = df_kafka \
    .withColumn("schema_id", get_schema_id("raw_bytes")) \
    .withColumn("payload",   get_payload("raw_bytes")) \
    .withColumn("order_id",  decode_order_id("payload"))

print("Parsed Kafka Protobuf messages:")
df_parsed.select("schema_id","order_id").show(5)
print(f"\nTotal: {df_parsed.count():,} messages | schema_id=1: {df_parsed.filter(F.col('schema_id')==1).count():,}")


## Step 4 — Full Deserialization Pipeline



In [ ]:

# Complete pipeline: Kafka bytes → Spark DataFrame → Parquet
import pathlib

PROTO_DIR = f"{DATA_DIR}/protos"

# Try using from_protobuf if descriptor is available
DESC_FILE = f"{PROTO_DIR}/orders.desc"
if pathlib.Path(DESC_FILE).exists():
    try:
        from pyspark.sql.protobuf.functions import from_protobuf
        df_orders = df_kafka \
            .withColumn("payload", get_payload("raw_bytes")) \
            .select(
                from_protobuf(F.col("payload"), "Order", DESC_FILE).alias("o")
            ).select("o.*")
        print(f"Deserialized with from_protobuf: {df_orders.count():,} rows")
        df_orders.show(5)
    except Exception as e:
        print(f"from_protobuf error: {e}")
        print("Falling back to manual UDF deserialization")

# Write to Parquet
df_parsed.select("schema_id","order_id","payload") \
         .write.mode("overwrite") \
         .parquet(f"{DATA_DIR}/kafka_protobuf_landing")
print(f"Saved to landing Parquet: {spark.read.parquet(f'{DATA_DIR}/kafka_protobuf_landing').count():,} rows")


## Summary



In [ ]:

print("""
Kafka + Protobuf in Spark:

  1. Read raw bytes from Kafka
     raw = spark.readStream.format("kafka")...load()
     # value column = BinaryType (serialized Protobuf)

  2. Deserialize with from_protobuf (Spark 4.0+)
     from pyspark.sql.protobuf.functions import from_protobuf
     df.select(
         from_protobuf(col("value"), "Order", "/path/orders.desc").alias("order")
     ).select("order.*")

  3. Confluent Schema Registry format (magic byte prefix)
     # Strip 5-byte header (0x00 + 4-byte schema_id) before from_protobuf
     payload = expr("substring(value, 6, length(value))")  # skip first 5 bytes
     df.select(from_protobuf(payload, "Order", desc_path).alias("order"))

  4. Write to Delta/Parquet
     .writeStream.format("delta").option("path","/data/orders/").start()
""")
